# Tái lập MDBGF trên Colab

Chạy lần lượt từ trên xuống. Cần Colab Pro với GPU A100 hoặc L4 (Runtime → Change runtime type).
Checkpoint lưu trên Google Drive, nên phiên bị ngắt thì chỉ cần chạy lại từ ô 1 tới ô chạy chính.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Gắn Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_ROOT = '/content/drive/MyDrive/mdbgf-runs'

## 3. Lấy mã nguồn

Mã tác giả được cố định ở commit `c512d79`.

In [ ]:
REPO_URL = 'https://github.com/HatDuaa/DLDT.git'
REPRO = '/content/DLDT/bao-cao-seminar-MMKGC/MDBGF/reproduce'
SRC = '/content/MDBGF'
!rm -rf {SRC} /content/DLDT
!git clone -q https://github.com/ferryman-ship/MDBGF.git {SRC} && git -C {SRC} checkout -q c512d79
!git clone -q --depth 1 {REPO_URL} /content/DLDT

## 4. Cài thư viện và chuẩn bị dữ liệu

In [ ]:
!pip install -q -r {REPRO}/requirements.txt
!python {REPRO}/prepare_data.py --src {SRC} --data MKG-W

## 5. Đăng nhập WandB

Thêm API key vào Colab Secrets (biểu tượng chìa khoá bên trái) với tên `WANDB_API_KEY`.

In [ ]:
import wandb
from google.colab import userdata
wandb.login(key=userdata.get('WANDB_API_KEY'))

## 6. Chạy thử 10 epoch

Để đo thời gian mỗi epoch và bộ nhớ GPU trước khi chạy thật. Lần chạy này không ghi lên WandB.

In [ ]:
!rm -rf {OUT_ROOT}/smoke
!python {REPRO}/train.py --src {SRC} --out {OUT_ROOT}/smoke --num_epoch 10 --valid_epoch 5 --no_wandb

## 7. Chạy chính

Chọn một dòng trong `RUNS`. Mỗi lần chạy dừng sau `--max_hours` giờ và lưu checkpoint;
chạy lại đúng ô này (sau các ô 1–5), giữ nguyên `RUN` và `EPOCHS`, để tiếp tục từ checkpoint cuối.

In [ ]:
RUNS = {
    'main_code':  '--md code',
    'main_paper': '--md paper',
    'no_md':      '--md none',
    'no_bgf':     '--no_bgf',
    'no_ahns':    '--mu 0',
}
RUN = 'main_code'
EPOCHS = 1550  # số epoch nhóm dùng trong báo cáo (bài báo: 5000)
!python {REPRO}/train.py --src {SRC} --out {OUT_ROOT}/{RUN} --name {RUN} {RUNS[RUN]} --num_epoch {EPOCHS} --max_hours 11